# 3 - Assemble Clinical and Molecular Metadata

**Purpose**: Build the master metadata table combining clinical outcomes,
tumour purity/TMB/MSI from PURPLE, and RNA immune signatures from Isofox.
Assigns broad tumour-type cohort labels for CPI (SKCM, NSCLC, BLCA, CRC-MSI, Other)
and pan-database cohorts (BRCA, LUNG, CRC, etc.).

| | |
|---|---|
| **Inputs** | `READY_DIR/clinical_go.txt`, `PURPLE_DIR/purity.csv`, `PURPLE_DIR/qc.csv`, `CIDER_PROJ_DIR/isofox_genesets_ready_log2.csv`, `PAIRED_IDS` |
| **Outputs** | `CIDER_PROJ_DIR/meta_base.csv` |
| **Run after** | `0_collect.ipynb`, `2_rna.ipynb` |
| **Run before** | `4_base.ipynb` |

In [ ]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))
source(paste0(dirname(getwd()), "/helpers/config.r"))

assert_files_exist(
  "Clinical data"    = paste0(READY_DIR, "clinical_go.txt"),
  "Paired RNA/DNA"   = PAIRED_IDS,
  "Purity table"     = paste0(PURPLE_DIR, "purity.csv"),
  "QC table"         = paste0(PURPLE_DIR, "qc.csv"),
  "RNA gene sets"    = paste0(CIDER_PROJ_DIR, "isofox_genesets_ready_log2.csv")
)

In [8]:
#fread("/mnt/immunocompnas1/datasets/hartwig_hg38/biomarkers/database/lilac/lilac.csv")

In [ ]:
clinical <-
  fread(paste0(READY_DIR, "clinical_go.txt")) |>
  transmute(
    sampleId,
    donorId,
    age               = age,
    sex               = sex,
    priorRadiotherapy = ifelse(hasRadiotherapyPreTreatment == "NULL", NA, hasRadiotherapyPreTreatment),
    pretreated        = ifelse(hasSystemicPreTreatment == "NULL", NA, hasSystemicPreTreatment),    
    timeBiopsyToTreatmentDays,
    # Biopsy site. biopsyStructureRaw preserves the original clinical_go.txt
    # string ("NULL" mapped to NA) unmodified, for tables that need finer
    # detail than the regression covariate below (e.g. Table S1,
    # 1_figures/11f_biopsy_location_breakdown.ipynb). biopsyStructure then
    # collapses it to BIOPSY_STRUCTURE_LEVELS (helpers/config.r) for use as
    # a low-cardinality regression covariate in 5_models.ipynb — collapsed
    # once here (rather than per-consuming-notebook) since that grouping
    # has one purpose and should be identical everywhere it's used.
    biopsyStructureRaw = ifelse(biopsyStructure == "NULL", NA_character_, biopsyStructure),
    # "Skin (subcutaneous)" and "Skin (cutaneous)" are pooled into one
    # "Skin (subcutaneous/cutaneous)" level (BIOPSY_STRUCTURE_LEVELS,
    # helpers/config.r) -- everything else passes through unchanged via
    # the identity branch below.
    biopsyStructure = factor(
      case_when(
        biopsyStructure == "NULL" ~ NA_character_,
        biopsyStructure %in% c("Skin (subcutaneous)", "Skin (cutaneous)") ~ "Skin (subcutaneous/cutaneous)",
        biopsyStructure %in% c("Lymph node", "Liver", "Lung") ~ biopsyStructure,
        TRUE ~ "Other"
      ),
      levels = BIOPSY_STRUCTURE_LEVELS
    ),
    primaryTumorLocation,
    primaryTumorType,
    treatment,
    # clinical_go.txt encodes missing values as the literal string "NULL"
    # (see priorRadiotherapy/pretreated/biopsyStructureRaw above) -- mechanism
    # was the one field in this block that skipped that conversion, so any
    # patient with a genuinely unknown mechanism carried the string "NULL"
    # through as if it were a real category instead of NA. That's what let
    # them silently fail to match any of 5_models.ipynb's MECHANISM_LEVELS
    # and show up as an unlabeled NA row in the mechanism forest plot instead
    # of being cleanly excluded like every other "NULL"-coded field.
    mechanism = ifelse(
      treatment %in% c("Ipilimumab/Nivolumab", "Nivolumab/Ipilimumab"),
      "Anti-CTLA4/Anti-PD1",
      ifelse(mechanism == "NULL", NA, mechanism)
    ),
    cpi       = (type == "Immunotherapy" | grepl("Nivolumab", treatment) | grepl("Pembrolizumab", treatment) |
                 grepl("Atezolizumab", treatment) | grepl("Durvalumab", treatment) |
                 grepl("Avelumab", treatment) | grepl("Tremelimumab", treatment) | grepl("PD-L1 antibody", treatment)) &
               (is.na(timeBiopsyToTreatmentDays) | timeBiopsyToTreatmentDays > -1),
    dcb       = response,  # Durable Clinical Benefit as defined in clinical_go.txt (includes SD ≥6 months)
    response  = bestOverallResponse,  # Best overall response: PR/CR only
    responder = ifelse(response == 1, "Responder", "Non-Responder"),
    osEvent,
    osDays,
    pfsEvent,
    pfsEventDays
  ) |>
  group_by(donorId) |>
  slice_min(timeBiopsyToTreatmentDays, n = 1, with_ties = FALSE) |>
  ungroup()
  # Note: CPI filtering and response drop_na are applied in 4_base.ipynb
  # when producing cpi_ready.rds. meta_base retains all patients.

In [ ]:
# Assigns cohort_broad to each sample based on tumor type and location.
# Five fixed groups: SKCM | NSCLC | BLCA | CRC-MSI | Other
#
# CRC is split on msi_high (from the dna table; must be present in df).
# All other tumour types collapse to "Other".
assign_immunotherapy_cohorts <- function(df) {
  df |>
    mutate(
      primaryTumorLocation = trimws(primaryTumorLocation),
      primaryTumorType     = trimws(primaryTumorType),

      cohort_broad = case_when(
        # SKCM — Skin Cutaneous Melanoma
        primaryTumorType == "Melanoma" &
          primaryTumorLocation %in% c("Skin", "Cancer of unknown primary (CUP)") ~ "SKCM",

        # NSCLC — Non-Small Cell Lung Cancer
        primaryTumorLocation == "Lung" &
          (grepl("Non-small cell", primaryTumorType) |
           primaryTumorType %in% c("Other", "Small cell carcinoma (SCLC)")) ~ "NSCLC",

        # BLCA — Bladder / Urothelial Carcinoma
        primaryTumorType == "Urothelial carcinoma" |
          (primaryTumorLocation %in% c("Bladder", "Ureter/renal pelvis", "Urethra") &
           primaryTumorType == "Unknown") ~ "BLCA",

        # CRC-MSI — Colorectal, MSI-High only
        primaryTumorLocation %in% c("Colon", "Rectum") &
          primaryTumorType %in% c("Unknown", "Adenocarcinoma",
                                  "Mucinous adenocarcinoma", "Other") &
          msi_high == 1 ~ "CRC-MSI",

        TRUE ~ "Other"
      )
    )
}


# =============================================================================
# assign_pan_cohorts() — cohort assignment for the full Hartwig database.
# Used for Figure 2A landscape validation and Figure 7's predicted response
# landscape (all patients, not just ICI-treated).
# 22 named cohorts ordered by N descending; everything else → "Other".
#
# Cohort acronyms follow TCGA conventions where applicable.
# CRC is split on msi_high: CRC-MSS vs CRC-MSI. MSI-high patients from every
# OTHER tissue are pooled into "Non-CRC MSI" rather than left in their
# tissue-of-origin bucket — MSI-high tumours are highly immunogenic largely
# independent of tissue of origin (cf. pembrolizumab's tissue-agnostic
# MSI-H/dMMR approval), so mixing a handful of MSI-high outliers into e.g.
# NSCLC or SKCM would confound those cohorts' baseline immunogenicity.
# This rule is checked immediately after CRC-MSI and before every
# tissue-of-origin rule below, so every other named cohort is implicitly
# MSI-low/MSI-unknown only.
# =============================================================================
assign_pan_cohorts <- function(df) {
  df |>
    mutate(
      primaryTumorLocation = trimws(primaryTumorLocation),
      primaryTumorType     = trimws(primaryTumorType),

      cohort_pan = case_when(
        # BRCA — Breast Cancer
        primaryTumorLocation == "Breast" ~ "BRCA",

        # NSCLC — Non-Small Cell Lung Cancer
        primaryTumorLocation == "Lung" &
          grepl("Non-small cell|Other|Unknown", primaryTumorType) ~ "NSCLC",

        # LNET — Lung Neuroendocrine: SCLC + NET + NEC + carcinoid
        primaryTumorLocation == "Lung" &
          grepl("Small cell|Neuroendocrine|carcinoid", primaryTumorType) ~ "LNET",

        # CRC-MSI — Colorectal, MSI-High; before CRC to take priority
        primaryTumorLocation %in% c("Colon", "Rectum") &
          msi_high == 1 ~ "CRC-MSI",

        # Non-CRC MSI — MSI-High, any other tissue; checked before all
        # tissue-of-origin rules below so they become MSI-low/unknown only.
        # NA msi_high falls through to the tissue rules unaffected (same
        # behaviour as the CRC-MSI rule above).
        msi_high == 1 ~ "Non-CRC MSI",

        # CRC-MSS — Colorectal, MSS
        primaryTumorLocation %in% c("Colon", "Rectum") &
          msi_high == 0 ~ "CRC-MSS",

        # PRAD — Prostate
        primaryTumorLocation == "Prostate" ~ "PRAD",

        # SARC — Soft Tissue Sarcoma
        primaryTumorLocation == "Soft tissue" ~ "SARC",

        # SKCM — Skin
        primaryTumorLocation == "Skin" ~ "SKCM",

        # OV — Ovarian Cancer
        primaryTumorLocation == "Ovary" ~ "OV",

        # CNS — Brain / Central Nervous System
        primaryTumorLocation == "Brain/central nervous system" ~ "CNS",

        # PAAD — Pancreatic Cancer
        primaryTumorLocation == "Pancreas" ~ "PAAD",

        # KIRC — Kidney / Renal Cell Carcinoma
        primaryTumorLocation %in% c("Kidney", "Ureter/renal pelvis") ~ "KIRC",

        # ESOG — Esophageal / Gastroesophageal Junction
        primaryTumorLocation == "Esophagus/gastroesophageal junction" ~ "ESOG",

        # BLCA — Bladder / Urothelial Carcinoma
        primaryTumorLocation %in% c("Bladder", "Urethra") ~ "BLCA",

        # CHOL — Biliary Tract / Cholangiocarcinoma
        primaryTumorLocation == "Biliary tract" ~ "CHOL",

        # STAD — Stomach / Gastric Adenocarcinoma
        primaryTumorLocation == "Stomach" ~ "STAD",

        # CESC — Cervical Cancer
        primaryTumorLocation == "Cervix" ~ "CESC",

        # ENDO — Endometrial / Uterine Cancer
        primaryTumorLocation == "Endometrium/uterus" ~ "ENDO",

        # MESO — Mesothelioma (pleura, peritoneum, unspecified)
        primaryTumorLocation %in% c("Pleura", "Peritoneum") |
          primaryTumorType == "Mesothelioma" ~ "MESO",

        # LIHC — Liver Hepatocellular Carcinoma
        primaryTumorLocation == "Liver" ~ "LIHC",

        # CUP — Cancer of Unknown Primary
        primaryTumorLocation == "Cancer of unknown primary (CUP)" ~ "CUP",

        # Other — all remaining tumour types
        TRUE ~ "Other"
      )
    )
}

In [4]:
matching <-
  fread(PAIRED_IDS) |>
  mutate(sampleId = sampleIdRNA)

In [ ]:
# Read purity estimates from collected PURPLE purity file.
# tmbPerMb is kept RAW here (previously pre-transformed to tmb = log(tmbPerMb+1)
# at this exact point, which meant every downstream notebook had to either
# divide by log(10) to get log10, or exp(tmb)-1 to recover the raw value --
# fragile, and already caused a real double-log bug in 10_unique_vs_total.ipynb
# (scale(log1p(tmb)), logging an already-logged value). Transforms are now
# applied explicitly at each point of use instead (log2(tmbPerMb + 1), the
# same convention used for CIDER and RNA -- see 9_roc.ipynb).
purity_raw <-
  fread(paste0(PURPLE_DIR, "purity.csv")) |>
  transmute(
    sampleId,
    purity,
    tmbPerMb,
    tmb_high   = as.integer(tmbStatus == "HIGH"),
    msi_high   = as.integer(msStatus  == "MSI")
  )

# Read QC flags from collected PURPLE QC file and join to purity.
# full_join retains all purity samples; QCStatus defaults to PASS if no QC record exists.
dna <- fread(paste0(PURPLE_DIR, "qc.csv")) |>
  select(sampleId, QCStatus, AmberMeanDepth) |>
  full_join(purity_raw, by = "sampleId") |>
  mutate(QCStatus = ifelse(is.na(QCStatus), "PASS", QCStatus)) |>
  # Keep only samples that passed QC or have low purity (biologically valid)
  filter(QCStatus %in% c("PASS", "WARN_LOW_PURITY"))

In [ ]:
rna <-
  fread(paste0(CIDER_PROJ_DIR, "isofox_genesets_ready_log2.csv")) |>
  inner_join(matching, by = "sampleId") |>
  transmute(
    sampleId          = sampleIdDNA,
    tcell_tpm,
    # Mean-aggregated CD8 Effector (tGE8) score and PD-L1 (CD274) expression
    # -- additional RNA comparators for supplemental DNA-vs-RNA analyses, not
    # a replacement for tcell_tpm.
    rna_teff_score,
    rna_pdl1_score,
    # Second, independent T-cell presence signature (CD3D/CD3E/CD3G/CD2/TRAC)
    # from the Human Cell Landscape atlas (Han, X., Zhou, Z., Fei, L. et al.
    # Construction of a human cell landscape at single-cell level. Nature
    # 581, 303-309, 2020) — a fourth RNA comparator for
    # 19_dna_vs_rna_response.ipynb, not a replacement for tcell_tpm.
    rna_tcell_hcl_score,
    # Checkpoint/exhaustion panel -- independently-verified gene-set
    # comparator for the supplemental gene-set validation figure
    # (1_figures/21_tcell_geneset_panel.ipynb) -- see 2_rna.ipynb for exact
    # gene list and citations.
    rna_exhaustion_score,
    # T-cell-inflamed GEP (Ayers et al. 2017) -- unlike the other comparators
    # above, this one is a secondary-source reconstruction, not verified
    # against the original paper's own supplementary table (see
    # 2_rna.ipynb's panel definition and
    # fig_references/tcell_signatures_literature.md for the caveat).
    rna_ayers_geps_score
  )


In [9]:
meta_base <-
  clinical |>
  inner_join(dna, by = "sampleId") |>  # inner: drops samples with unreliable purity
  left_join(rna, by = "sampleId") |>
  relocate(sampleId, primaryTumorLocation, primaryTumorType)

- Assign cohorts

In [10]:
meta_base <- meta_base |>
  assign_immunotherapy_cohorts() |>
  assign_pan_cohorts()

# 2 - Send it

In [11]:
fwrite(meta_base, paste0(CIDER_PROJ_DIR, "meta_base.csv"))